<a href="https://colab.research.google.com/github/elangbijak4/Swarm-Humanoid-Control-PoC/blob/main/Implementation_code2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Extended Numerical Evaluation for Paper Tables
# No additional plots generated. Pure numeric outputs for direct insertion into tables.

import numpy as np
import hashlib
import random

np.random.seed(123)

def run_simulation(N, T=200, disturbance_step=50):
    alpha, beta, gamma = 0.05, 0.02, 0.01

    states_swarm = np.random.randn(N, 2)
    states_central = states_swarm.copy()

    def swarm_update(states):
        new_states = states.copy()
        global_mean = np.mean(states, axis=0)
        for i in range(N):
            alignment = np.mean(states - states[i], axis=0)
            cohesion = global_mean - states[i]
            stability = -states[i]
            update = alpha * alignment + beta * cohesion + gamma * stability
            new_states[i] += update
        return new_states

    def centralized_update(states):
        global_mean = np.mean(states, axis=0)
        return states + 0.1 * (global_mean - states)

    swarm_history = []
    central_history = []

    for t in range(T):
        if t == disturbance_step:
            states_swarm[0] += np.array([5.0, -5.0])
            states_central[0] += np.array([5.0, -5.0])

        states_swarm = swarm_update(states_swarm)
        states_central = centralized_update(states_central)

        swarm_history.append(states_swarm.copy())
        central_history.append(states_central.copy())

    swarm_history = np.array(swarm_history)
    central_history = np.array(central_history)

    def stability_index(history):
        return np.mean(np.linalg.norm(history, axis=2))

    def recovery_time(history):
        norms = [np.mean(np.linalg.norm(history[t], axis=1)) for t in range(T)]
        threshold = norms[0] * 0.2
        for i in range(disturbance_step, T):
            if norms[i] < threshold:
                return i - disturbance_step
        return T - disturbance_step

    return (
        stability_index(swarm_history),
        stability_index(central_history),
        recovery_time(swarm_history),
        recovery_time(central_history)
    )

# -----------------------------
# 1. Scalability Results
# -----------------------------

agent_sizes = [10, 20, 50, 100]
results = {}

for N in agent_sizes:
    SI_swarm, SI_central, RT_swarm, RT_central = run_simulation(N)
    results[N] = (SI_swarm, SI_central, RT_swarm, RT_central)

print("=== Scalability Results ===")
print("N | SI_Swarm | SI_Central | Recovery_Swarm | Recovery_Central")
for N in agent_sizes:
    SI_swarm, SI_central, RT_swarm, RT_central = results[N]
    print(f"{N} | {SI_swarm:.4f} | {SI_central:.4f} | {RT_swarm} | {RT_central}")

# -----------------------------
# 2. Malicious Rule Attack Test
# -----------------------------

def hash_parameters(alpha, beta, gamma):
    param_string = f"{alpha}-{beta}-{gamma}"
    return hashlib.sha256(param_string.encode()).hexdigest()

N = 50
consensus_threshold = 0.7

# Malicious proposal (extreme parameters)
mal_alpha, mal_beta, mal_gamma = 1.0, 1.0, 1.0
mal_hash = hash_parameters(mal_alpha, mal_beta, mal_gamma)

# Simulate cautious voting (low approval tendency)
votes = [random.random() > 0.8 for _ in range(N)]  # Only ~20% approve
approval_ratio = sum(votes) / N

print("\n=== Malicious Rule Attack Simulation ===")
print("Proposed Hash (truncated):", mal_hash[:20], "...")
print("Approval Ratio:", round(approval_ratio, 2))
print("Rule Accepted?" , approval_ratio >= consensus_threshold)


=== Scalability Results ===
N | SI_Swarm | SI_Central | Recovery_Swarm | Recovery_Central
10 | 0.3698 | 0.6243 | 87 | 150
20 | 0.2502 | 0.4188 | 36 | 150
50 | 0.1969 | 0.3664 | 15 | 150
100 | 0.1297 | 0.2194 | 0 | 6

=== Malicious Rule Attack Simulation ===
Proposed Hash (truncated): 0feb910cd691b1a5614b ...
Approval Ratio: 0.14
Rule Accepted? False
